# US06 – Evolução histórica da renda por região, por estado e comparação entre estados

**Como** analista de crédito, **quero** visualizar a evolução histórica dos dados financeiros e econômicos, **para** identificar a trajetória de melhora ou piora no perfil da região.

**Fonte:** SIDRA/IBGE, tabela 7437 – rendimento médio mensal real da população residente com rendimento (Grandes Regiões e Unidades da Federação).

**Como usar:** rode as duas células e use os botões:
1. **Região:** "Todas as regiões" ou uma região específica.
2. **Visão:** "Região em geral", "Estados da região" ou "Comparação entre estados" (região em linha grossa, estados em linhas finas).
3. **Estado:** "Todos os estados" ou um estado específico (a lista muda conforme a região escolhida).

| Critério de aceite | Onde está |
|---|---|
| Filtro de dados por região | Botões "Região" (e "Estado") |
| Gráfico de linhas para comparativo de evolução | Gráfico |
| Dados de no mínimo 2 anos | Verificado na célula 1 |

In [3]:
import requests
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
import ipywidgets as widgets
from IPython.display import display

URL_REGIOES = "https://apisidra.ibge.gov.br/values/t/7437/n2/all/v/all/p/all"  # n2 = Grandes Regiões
URL_ESTADOS = "https://apisidra.ibge.gov.br/values/t/7437/n3/all/v/all/p/all"  # n3 = Unidades da Federação


REGIAO_POR_DIGITO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}


def carregar(url):
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    raw = resp.json()
    chave = {descricao: col for col, descricao in raw[0].items()}
    d = pd.DataFrame(raw[1:]).rename(columns={
        "D1N": "territorio",
        "D1C": "codigo",
        chave["Variável"]: "variavel",
        chave["Ano"]: "ano",
        chave["Tipo de rendimento"]: "tipo_rendimento",
        "MN": "unidade",
        "V": "valor",
    })
    d = d[["territorio", "codigo", "variavel", "tipo_rendimento", "unidade", "ano", "valor"]]
    d["ano"] = d["ano"].astype(int)
    d["valor"] = pd.to_numeric(d["valor"], errors="coerce")
    return d.dropna(subset=["valor"])


df_reg = carregar(URL_REGIOES)
df_uf = carregar(URL_ESTADOS)


regioes = list(df_reg[["territorio", "codigo"]].drop_duplicates().sort_values("codigo")["territorio"])
_ufs = df_uf[["territorio", "codigo"]].drop_duplicates().sort_values("codigo")
ufs_ordem = list(_ufs["territorio"])
uf_regiao = {r.territorio: REGIAO_POR_DIGITO[str(r.codigo)[0]] for r in _ufs.itertuples()}


anos = sorted(set(df_reg["ano"]) & set(df_uf["ano"]))
assert len(anos) >= 2, "A US exige no mínimo 2 anos de dados."
print(f"Regiões: {len(regioes)} | Estados: {len(ufs_ordem)} | Anos: {len(anos)} ({anos[0]} a {anos[-1]})")

Regiões: 5 | Estados: 27 | Anos: 14 (2012 a 2025)


In [4]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from cycler import cycler

TODAS = "Todas as regiões"
TODOS = "Todos os estados"
V_REG, V_UF, V_COMP = "Regiões", "Estados", "Comparar estados"


variaveis = list(df_reg.loc[df_reg["unidade"] == "Reais", "variavel"].unique())
tipos = list(df_reg["tipo_rendimento"].unique())
padrao = next((v for v in variaveis if "último ano" in v), variaveis[0])


w_visao = widgets.ToggleButtons(
    options=[V_REG, V_UF, V_COMP],
    value=V_REG,
    description="Visualização:",
    style={"description_width": "initial"}
)
w_regiao = widgets.Dropdown(
    options=[TODAS] + regioes, value=TODAS, description="Região:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="320px")
)
w_estado = widgets.Dropdown(
    options=[TODOS] + ufs_ordem, value=TODOS, description="Estado:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="320px")
)
w_comparar = widgets.SelectMultiple(
    options=ufs_ordem,
    value=tuple(u for u in ["São Paulo", "Bahia"] if u in ufs_ordem),
    description="Estados:",
    rows=8, style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)
w_var = widgets.Dropdown(
    options=variaveis, value=padrao, description="Indicador:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="100%")
)
w_tipo = widgets.Dropdown(
    options=tipos, description="Tipo de rendimento:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px")
)

btn_todos = widgets.Button(description="Selecionar todos os estados", icon="check")
btn_limpar = widgets.Button(description="Limpar seleção", icon="times")
saida = widgets.Output()

def atualizar_estados(*_):
    regiao = w_regiao.value
    lista = [u for u in ufs_ordem if regiao == TODAS or uf_regiao[u] == regiao]
    w_estado.options = [TODOS] + lista
    if w_estado.value not in [TODOS] + lista:
        w_estado.value = TODOS

def atualizar_controles(*_):
    visao = w_visao.value
    w_regiao.layout.display = "block" if visao == V_UF else "none"
    w_estado.layout.display = "block" if visao == V_UF else "none"
    w_comparar.layout.display = "block" if visao == V_COMP else "none"
    btn_todos.layout.display = "inline-flex" if visao == V_COMP else "none"
    btn_limpar.layout.display = "inline-flex" if visao == V_COMP else "none"

def selecionar_todos(_):
    w_comparar.value = tuple(ufs_ordem)

def limpar_selecao(_):
    w_comparar.value = ()

w_regiao.observe(atualizar_estados, names="value")
w_visao.observe(atualizar_controles, names="value")
btn_todos.on_click(selecionar_todos)
btn_limpar.on_click(limpar_selecao)
atualizar_estados()
atualizar_controles()

def pivotar(dados, ordem, variavel, tipo):
    filtro = dados[(dados["variavel"] == variavel) &
                   (dados["tipo_rendimento"] == tipo)]
    return (filtro.pivot_table(index="ano", columns="territorio",
                               values="valor", aggfunc="mean")
            .reindex(columns=ordem).sort_index()
            .dropna(axis=1, how="all"))

def desenhar(regiao, visao, estado, estados_comparar, variavel, tipo):
    with saida:
        clear_output(wait=True)
        unidade = df_reg.loc[df_reg["variavel"] == variavel, "unidade"].iloc[0]
        t_reg = pivotar(df_reg, regioes, variavel, tipo)
        t_uf = pivotar(df_uf, ufs_ordem, variavel, tipo)

        if visao == V_REG:
            tab = t_reg
            titulo = "Evolução histórica por região"
        elif visao == V_UF:
            lista = [u for u in ufs_ordem
                     if regiao == TODAS or uf_regiao[u] == regiao]
            if estado != TODOS:
                lista = [estado] if estado in lista else []
            tab = t_uf.reindex(columns=lista).dropna(axis=1, how="all")
            titulo = "Evolução histórica por estado"
            if regiao != TODAS:
                titulo += f" – {regiao}"
        else:
            selecionados = list(estados_comparar)
            if len(selecionados) < 2:
                print("Selecione pelo menos dois estados para comparar.")
                return
            tab = t_uf.reindex(columns=selecionados).dropna(axis=1, how="all")
            titulo = "Comparação da evolução histórica entre estados"

        if tab.empty or tab.dropna(how="all").empty:
            print("Não há dados disponíveis para os filtros escolhidos.")
            return

        fig, ax = plt.subplots(figsize=(12, 6.5))
        cores = plt.cm.tab20.colors
        ax.set_prop_cycle(cycler(color=cores))
        for col in tab.columns:
            ax.plot(tab.index, tab[col], marker="o",
                    markersize=4 if len(tab.columns) > 10 else 5,
                    linewidth=1.7 if len(tab.columns) > 10 else 2.3,
                    label=col)
        ax.set_title(f"{titulo}\n{variavel}", fontsize=12, pad=14)
        ax.set_xlabel("Ano")
        ax.set_ylabel(f"Rendimento ({unidade})")
        ax.grid(True, alpha=0.25)
        ax.set_xticks(list(tab.index))
        ax.tick_params(axis="x", rotation=35)
        ax.spines[["top", "right"]].set_visible(False)
        if len(tab.columns) > 10:
            ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5),
                      fontsize=8, ncol=1, frameon=False)
        else:
            ax.legend(loc="best", fontsize=9, frameon=False)
        fig.tight_layout()
        plt.show()


        tab_valida = tab.dropna(how="all")
        primeiro, ultimo = tab_valida.index.min(), tab_valida.index.max()
        resumo = pd.DataFrame({
            f"Valor {primeiro}": tab.loc[primeiro],
            f"Valor {ultimo}": tab.loc[ultimo]
        })
        resumo["Variação (R$)"] = resumo[f"Valor {ultimo}"] - resumo[f"Valor {primeiro}"]
        resumo["Variação (%)"] = (
            resumo[f"Valor {ultimo}"] / resumo[f"Valor {primeiro}"] - 1
        ) * 100
        resumo["Trajetória"] = resumo["Variação (%)"].apply(
            lambda x: "Aumento" if x > 0 else ("Redução" if x < 0 else "Estável"))
        resumo.index.name = "Território"
        resumo = resumo.sort_values("Variação (%)", ascending=False)
        print(f"Resumo da evolução: {primeiro} a {ultimo}")
        display(resumo.style.format({
            f"Valor {primeiro}": "R$ {:,.0f}",
            f"Valor {ultimo}": "R$ {:,.0f}",
            "Variação (R$)": "R$ {:+,.0f}",
            "Variação (%)": "{:+.1f}%"
        }, na_rep="—"))

        print("Valores históricos por ano:")
        historico = tab.T
        historico.index.name = "Território"
        historico.columns.name = "Ano"
        display(historico.style.format("R$ {:,.0f}", na_rep="—"))

controles = widgets.VBox([
    w_visao, w_regiao, w_estado, w_comparar,
    widgets.HBox([btn_todos, btn_limpar]), w_var, w_tipo
])
display(controles, saida)

for controle in [w_visao, w_regiao, w_estado, w_comparar, w_var, w_tipo]:
    controle.observe(lambda change: desenhar(
        w_regiao.value, w_visao.value, w_estado.value, w_comparar.value,
        w_var.value, w_tipo.value
    ), names="value")
desenhar(w_regiao.value, w_visao.value, w_estado.value, w_comparar.value,
         w_var.value, w_tipo.value)


Output()